<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [31]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    X, y = fetch_openml('mnist_784', version=1, as_frame=False, return_X_y=True)
    y = y.astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    return X_train, X_test, y_train, y_test

Não é sempre necessario fazer a normalizaçao dos dados pois vai depender do medolo que vai ser ultilizado.
Tanto o Random Forest quanto o AdaBoost (que serão usados a seguir na atividade) não são sensíveis à escala dos dados. Esses modelos realizam divisões baseadas em limiares dos atributos, e não em distâncias, o que torna a normalização desnecessária.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [32]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42, n_estimators=100):
    
    clf = RandomForestClassifier(n_estimators=n_estimators, random_state=seed, n_jobs=-1)
    clf.fit(X_train, y_train)
    return clf

def train_adaboost(X_train, y_train, seed=42, n_estimators=50):
    
    clf = AdaBoostClassifier(n_estimators=n_estimators, random_state=seed)
    clf.fit(X_train, y_train)
    return clf

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [33]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):

    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [34]:
def run_pipeline(model_type="rf", seed=42, rf_estimators=100, ab_estimators=50):
    
    X_train, X_test, y_train, y_test = load_data(seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed=seed, n_estimators=rf_estimators)
    elif model_type in ("ab", "ada", "adaboost"):
        model = train_adaboost(X_train, y_train, seed=seed, n_estimators=ab_estimators)
    else:
        raise ValueError("model_type must be 'rf' or 'ab'")

    acc = evaluate(model, X_test, y_test)
    return acc

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [41]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def evaluate_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'report': classification_report(y_test, y_pred)}

X_train, X_test, y_train, y_test = load_data(42)
rf = train_random_forest(X_train, y_train, seed=42)
ab = train_adaboost(X_train, y_train, seed=42)
rf_metrics = evaluate_metrics(rf, X_test, y_test)
ab_metrics = evaluate_metrics(ab, X_test, y_test)

print('Random Forest metrics:')
print('Accuracy:', rf_metrics['accuracy'])
print('Precision (macro):', rf_metrics['precision'])
print('Recall (macro):', rf_metrics['recall'])
print('F1 (macro):', rf_metrics['f1'])
print('Classification report: \n', rf_metrics['report'])

print('AdaBoost metrics:')
print('Accuracy:', ab_metrics['accuracy'])
print('Precision (macro):', ab_metrics['precision'])
print('Recall (macro):', ab_metrics['recall'])
print('F1 (macro):', ab_metrics['f1'])
print('Classification report: \n', ab_metrics['report'])

Random Forest metrics:
Accuracy: 0.9672142857142857
Precision (macro): 0.9670109349902696
Recall (macro): 0.9669239858573642
F1 (macro): 0.9669440596046025
Classification report: 
               precision    recall  f1-score   support

           0       0.97      0.99      0.98      1381
           1       0.98      0.98      0.98      1575
           2       0.96      0.97      0.97      1398
           3       0.96      0.96      0.96      1428
           4       0.97      0.96      0.96      1365
           5       0.97      0.96      0.96      1263
           6       0.97      0.98      0.98      1375
           7       0.97      0.97      0.97      1459
           8       0.96      0.96      0.96      1365
           9       0.94      0.94      0.94      1391

    accuracy                           0.97     14000
   macro avg       0.97      0.97      0.97     14000
weighted avg       0.97      0.97      0.97     14000

AdaBoost metrics:
Accuracy: 0.6681428571428571
Precision (ma

O random forest teve um desempenho inicial superior ao do adaBoost


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [43]:
X_train, X_test, y_train, y_test = load_data(7)
rf = train_random_forest(X_train, y_train, seed=7)
ab = train_adaboost(X_train, y_train, seed=7)
rf_metrics = evaluate_metrics(rf, X_test, y_test)
ab_metrics = evaluate_metrics(ab, X_test, y_test)

print('Random Forest metrics (seed 7):')
print('Accuracy:', rf_metrics['accuracy'])
print('Precision (macro):', rf_metrics['precision'])
print('Recall (macro):', rf_metrics['recall'])
print('F1 (macro):', rf_metrics['f1'])

print('AdaBoost metrics (seed 7):')
print('Accuracy:', ab_metrics['accuracy'])
print('Precision (macro):', ab_metrics['precision'])
print('Recall (macro):', ab_metrics['recall'])
print('F1 (macro):', ab_metrics['f1'])

print("\n---------------------------------\n")

X_train, X_test, y_train, y_test = load_data(42)
rf = train_random_forest(X_train, y_train, seed=42)
ab = train_adaboost(X_train, y_train, seed=42)
rf_metrics = evaluate_metrics(rf, X_test, y_test)
ab_metrics = evaluate_metrics(ab, X_test, y_test)

print('Random Forest metrics (seed 42):')
print('Accuracy:', rf_metrics['accuracy'])
print('Precision (macro):', rf_metrics['precision'])
print('Recall (macro):', rf_metrics['recall'])
print('F1 (macro):', rf_metrics['f1'])

print('AdaBoost metrics (seed 42):')
print('Accuracy:', ab_metrics['accuracy'])
print('Precision (macro):', ab_metrics['precision'])
print('Recall (macro):', ab_metrics['recall'])
print('F1 (macro):', ab_metrics['f1'])


Random Forest metrics (seed 42):
Accuracy: 0.9681428571428572
Precision (macro): 0.9677813937337447
Recall (macro): 0.9680626930765787
F1 (macro): 0.9678886729927537
AdaBoost metrics (seed 42):
Accuracy: 0.6115
Precision (macro): 0.6602259943903731
Recall (macro): 0.6084269383435826
F1 (macro): 0.6117159728324217

---------------------------------

Random Forest metrics (seed 7):
Accuracy: 0.9672142857142857
Precision (macro): 0.9670109349902696
Recall (macro): 0.9669239858573642
F1 (macro): 0.9669440596046025
AdaBoost metrics (seed 7):
Accuracy: 0.6681428571428571
Precision (macro): 0.683584850108614
Recall (macro): 0.6657169606056172
F1 (macro): 0.6674785949892348


Sim após mudar as seeds os resultados tambem tiveram uma pequena alteração

O experimento é reprodutível, pois o parâmetro random_state (seed) está sendo utilizado tanto na divisão dos dados (load_data) quanto no treinamento dos modelos (Random Forest e AdaBoost).

Isso garante que, ao executar o código novamente com a mesma seed, os resultados obtidos serão os mesmos.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [44]:
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = load_data(42)
rf = train_random_forest(X_train, y_train, seed=42)
rf_train_acc = accuracy_score(y_train, rf.predict(X_train))
rf_test_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Random Forest - train accuracy: {rf_train_acc:.4f}")
print(f"Random Forest - test  accuracy: {rf_test_acc:.4f}")

{'rf_train': rf_train_acc, 'rf_test': rf_test_acc}

Random Forest - train accuracy: 1.0000
Random Forest - test  accuracy: 0.9672


{'rf_train': 1.0, 'rf_test': 0.9672142857142857}

Mesmo com alta acurácia no teste (96%), ainda pode haver overfitting se o desempenho no treino for significativamente maior. Nesse caso, o modelo apresenta um leve sobreajuste, mas ainda mantém boa capacidade de generalização.

Entre os modelos, o Random Forest tende a sofrer mais com overfitting, pois utiliza múltiplas árvores profundas que podem se ajustar excessivamente aos dados. Já o AdaBoost, por utilizar modelos mais simples e aprendizado iterativo, tende a generalizar melhor.

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [46]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

X_train, X_test, y_train, y_test = load_data(42)

rf_estimators = [15, 30, 50]
ab_estimators = [15, 30, 50]

print("=== Random Forest ===")
for n in rf_estimators:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)
    
    metrics = evaluate_metrics(rf, X_test, y_test)
    
    print(f"\nn_estimators = {n}")
    print('Accuracy:', metrics['accuracy'])
    print('Precision:', metrics['precision'])
    print('Recall:', metrics['recall'])
    print('F1:', metrics['f1'])


print("\n=== AdaBoost ===")
for n in ab_estimators:
    ab = AdaBoostClassifier(n_estimators=n, random_state=42)
    ab.fit(X_train, y_train)
    
    metrics = evaluate_metrics(ab, X_test, y_test)
    
    print(f"\nn_estimators = {n}")
    print('Accuracy:', metrics['accuracy'])
    print('Precision:', metrics['precision'])
    print('Recall:', metrics['recall'])
    print('F1:', metrics['f1'])


=== Random Forest ===

n_estimators = 15
Accuracy: 0.954
Precision: 0.953656312471676
Recall: 0.9534785838154303
F1: 0.9535111087781936

n_estimators = 30
Accuracy: 0.9615714285714285
Precision: 0.9612475647841627
Recall: 0.9611935361519428
F1: 0.9611896104082753

n_estimators = 50
Accuracy: 0.9639285714285715
Precision: 0.9636916534187293
Recall: 0.9635808744871488
F1: 0.9636028882149317

=== AdaBoost ===

n_estimators = 15
Accuracy: 0.40514285714285714
Precision: 0.4470212734182259
Recall: 0.40157802208050636
F1: 0.3779736229792535

n_estimators = 30
Accuracy: 0.5735714285714286
Precision: 0.5967154600200734
Recall: 0.5681161985940524
F1: 0.5706284222555179

n_estimators = 50
Accuracy: 0.6681428571428571
Precision: 0.683584850108614
Recall: 0.6657169606056172
F1: 0.6674785949892348


A depender das alterações feitas a no numero de estimators o desenpenho pode apresentar uma grade diferença, aumentando consideravelmente o desenpenho quando aumentado

O AdaBoost aparentou ser mais sensivel a mudança de hiperparametros tendo um aumento bastante considreavel no seu desenpenho 

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1 - Não, a acurácia sozinha não é suficiente, especialmente em problemas com múltiplas classes como o Fashion MNIST. Ela não mostra como o modelo se comporta em cada classe individualmente, podendo mascarar erros importantes.

2 - O uso de random_state garante reprodutibilidade, permitindo obter os mesmos resultados em execuções repetidas. Além disso, testar diferentes seeds ajuda a verificar a estabilidade dos resultados.

3 - Um possível problema é a avaliação baseada em apenas uma divisão treino/teste, o que pode introduzir viés nos resultados. Outro problema é a ausência de validação cruzada ou ajuste sistemático de hiperparâmetros, o que pode levar a conclusões imprecisas sobre o desempenho dos modelos. Além disso, o underfitting observado em algumas configurações (como AdaBoost com poucos estimadores) pode comprometer a análise comparativa entre os modelos.

4 - O pipeline é parcialmente confiável, pois garante reprodutibilidade através do uso de random_state e segue uma estrutura clara de carregamento, treinamento e avaliação.

In [39]:
# TODO: implemente load_data